#### Setup & Imports

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import DataLoader

from src.utils import get_raw_dir
from src.dataset import PneumoniaDataset
from src.preprocessing import train_transform, val_transform
from src.model import build_model, count_trainable_params
from src.train import train_one_epoch, validate_one_epoch, train_model

#### Device & Data

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [3]:
RAW_DIR = get_raw_dir()
cache_dir = '../data/cache'

train_df = pd.read_csv('../data/splits/train.csv')
val_df = pd.read_csv('../data/splits/val.csv')

train_dataset = PneumoniaDataset(train_df, RAW_DIR / "stage_2_train_images",
                                  transform=train_transform, cache_dir=cache_dir)
val_dataset = PneumoniaDataset(val_df, RAW_DIR / "stage_2_train_images",
                                transform=val_transform, cache_dir=cache_dir)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

#### 1. Load DenseNet121 Pretrained on ImageNet

In [4]:
model = models.densenet121(weights='IMAGENET1K_V1')

In [5]:
print(model)

DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, bias=T

In [6]:
# Final layer is named classifier — maps DenseNet's 1024 features to
# ImageNet's 1000 classes. This is what needs replacing next.
print(model.classifier)

Linear(in_features=1024, out_features=1000, bias=True)


In [7]:
dummy_input = torch.randn(2, 3, 224, 224)  # matches transform output shape
output = model(dummy_input)
print(output.shape)  # expect torch.Size([2, 1000])

torch.Size([2, 1000])


#### 2. Replace Final Classification Layer

In [8]:
NUM_CLASSES = 3

In [9]:
model.classifier = nn.Linear(in_features=1024, out_features=NUM_CLASSES)

In [10]:
print(model.classifier)  # expect Linear(in_features=1024, out_features=3, ...)

Linear(in_features=1024, out_features=3, bias=True)


In [11]:
dummy_input = torch.randn(2, 3, 224, 224)
output = model(dummy_input)
print(output.shape)  # expect torch.Size([2, 3])

torch.Size([2, 3])


This new classifier layer is randomly initialized — it hasn't learned anything yet, unlike the rest of the network which carries ImageNet-pretrained weights. This is exactly why the freeze vs. fine-tune decision matters: the new layer always needs training, but whether the rest of the network gets updated too is the actual decision point.

#### 3. Freeze vs. Fine-tune — `build_model()`
`build_model()` and `count_trainable_params()` (`src/model.py`) wrap the steps above into a reusable, toggleable function. Confirming both variants build correctly and checking their trainable parameter counts before committing to either.

In [12]:
model_frozen_check = build_model(NUM_CLASSES, freeze_backbone=True)
model_finetune_check = build_model(NUM_CLASSES, freeze_backbone=False)

print("Frozen — trainable/total:", count_trainable_params(model_frozen_check))
print("Fine-tune — trainable/total:", count_trainable_params(model_finetune_check))

Frozen — trainable/total: (3075, 6956931)
Fine-tune — trainable/total: (6956931, 6956931)


In [13]:
dummy_input = torch.randn(2, 3, 224, 224)
for name, m in [("frozen", model_frozen_check), ("finetune", model_finetune_check)]:
    out = m(dummy_input)
    print(name, out.shape)

frozen torch.Size([2, 3])
finetune torch.Size([2, 3])


Which strategy actually performs best can't be judged from parameter counts alone — that's an empirical question, answered in Section 6 by training and comparing validation performance, not decided here.

#### 4. Loss Function — Weighted CrossEntropyLoss
Class weights offset the imbalance found in Phase 1 (Normal 33% / No Opacity 44% / Lung Opacity 23%), computed here from the train split specifically, since that's the distribution the model actually trains on.

In [14]:
class_counts = train_df['label'].value_counts().sort_index()
total = class_counts.sum()
class_weights = total / (len(class_counts) * class_counts)
class_weights_tensor = torch.tensor(class_weights.values, dtype=torch.float32)
print(class_weights_tensor)

tensor([1.0050, 0.7524, 1.4795])


In [15]:
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

In [16]:
# Sanity check with dummy logits/labels — confirms the loss runs correctly
dummy_logits = torch.randn(4, NUM_CLASSES)
dummy_labels = torch.tensor([0, 1, 2, 1])
loss = criterion(dummy_logits, dummy_labels)
print(loss.item())

0.7365232706069946


#### 5. Training Loop Sanity Check
`train_one_epoch`, `validate_one_epoch`, and `train_model` are graduated into `src/train.py` and imported above. One quick single-epoch run here confirms the full loop works end-to-end before committing to the longer multi-strategy comparison in Section 6.

In [17]:
sanity_model = build_model(NUM_CLASSES, freeze_backbone=True).to(device)
class_weights_tensor = class_weights_tensor.to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

sanity_optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, sanity_model.parameters()),
    lr=1e-3
)

In [18]:
train_loss, train_acc = train_one_epoch(sanity_model, train_loader, sanity_optimizer, criterion, device)
val_loss, val_acc = validate_one_epoch(sanity_model, val_loader, criterion, device)

In [19]:
print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

Train Loss: 0.8545, Train Acc: 0.5677
Val Loss: 0.7848, Val Acc: 0.6096


#### 6. Strategy Comparison — Frozen vs. Full Fine-tune vs. Progressive Unfreeze

In [20]:
EPOCHS_FROZEN = 5
EPOCHS_FINETUNE = 5
EPOCHS_PROGRESSIVE_STAGE1 = 3
EPOCHS_PROGRESSIVE_STAGE2 = 5

**Variant A — Frozen backbone**

In [21]:
model_frozen = build_model(NUM_CLASSES, freeze_backbone=True).to(device)
optimizer_frozen = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_frozen.parameters()), lr=1e-3
)
history_frozen = train_model(model_frozen, train_loader, val_loader,
                              optimizer_frozen, criterion, device, EPOCHS_FROZEN)

Epoch 1/5 — Train Loss: 0.8571, Train Acc: 0.5756 | Val Loss: 0.7887, Val Acc: 0.6553
Epoch 2/5 — Train Loss: 0.8223, Train Acc: 0.5943 | Val Loss: 0.7660, Val Acc: 0.6331
Epoch 3/5 — Train Loss: 0.8065, Train Acc: 0.6027 | Val Loss: 0.7801, Val Acc: 0.6244
Epoch 4/5 — Train Loss: 0.8092, Train Acc: 0.6040 | Val Loss: 0.7764, Val Acc: 0.6024
Epoch 5/5 — Train Loss: 0.8004, Train Acc: 0.6060 | Val Loss: 0.7667, Val Acc: 0.5904


**Variant B — Full fine-tune from scratch**

In [22]:
model_finetune = build_model(NUM_CLASSES, freeze_backbone=False).to(device)
optimizer_finetune = torch.optim.Adam(model_finetune.parameters(), lr=1e-4)
history_finetune = train_model(model_finetune, train_loader, val_loader,
                                optimizer_finetune, criterion, device, EPOCHS_FINETUNE)

Epoch 1/5 — Train Loss: 0.7075, Train Acc: 0.6578 | Val Loss: 0.6537, Val Acc: 0.6713
Epoch 2/5 — Train Loss: 0.6472, Train Acc: 0.6896 | Val Loss: 0.6485, Val Acc: 0.7068
Epoch 3/5 — Train Loss: 0.6220, Train Acc: 0.7016 | Val Loss: 0.6057, Val Acc: 0.7215
Epoch 4/5 — Train Loss: 0.5986, Train Acc: 0.7170 | Val Loss: 0.6032, Val Acc: 0.7090
Epoch 5/5 — Train Loss: 0.5794, Train Acc: 0.7251 | Val Loss: 0.5952, Val Acc: 0.7108


**Variant C — Progressive unfreeze** (frozen warm-up, then unfreeze + continue training the same model at a lower learning rate)

**Stage 1** — build the model with a frozen backbone and train just the classifier head for a few epochs, so it stabilizes before any pretrained weights are touched.

In [23]:
model_progressive = build_model(NUM_CLASSES, freeze_backbone=True).to(device)
optimizer_stage1 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_progressive.parameters()), lr=1e-3
)

**Stage 1 (train)** — run the frozen warm-up for `EPOCHS_PROGRESSIVE_STAGE1` epochs.

In [24]:
history_stage1 = train_model(model_progressive, train_loader, val_loader,
                              optimizer_stage1, criterion, device, EPOCHS_PROGRESSIVE_STAGE1)

Epoch 1/3 — Train Loss: 0.8616, Train Acc: 0.5688 | Val Loss: 0.7730, Val Acc: 0.5954
Epoch 2/3 — Train Loss: 0.8194, Train Acc: 0.5948 | Val Loss: 0.7582, Val Acc: 0.6234
Epoch 3/3 — Train Loss: 0.8103, Train Acc: 0.6011 | Val Loss: 0.7812, Val Acc: 0.6039


**Stage 2** - unfreeze everything, continue training the same model at a lower LR

In [ ]:
for param in model_progressive.parameters():
    param.requires_grad = True

In [26]:
optimizer_stage2 = torch.optim.Adam(model_progressive.parameters(), lr=1e-5)

In [ ]:
history_stage2 = train_model(model_progressive, train_loader, val_loader,
                              optimizer_stage2, criterion, device, EPOCHS_PROGRESSIVE_STAGE2)

Epoch 1/5 — Train Loss: 0.7438, Train Acc: 0.6350 | Val Loss: 0.7022, Val Acc: 0.6698
Epoch 2/5 — Train Loss: 0.6893, Train Acc: 0.6734 | Val Loss: 0.6684, Val Acc: 0.6766
Epoch 3/5 — Train Loss: 0.6669, Train Acc: 0.6829 | Val Loss: 0.6605, Val Acc: 0.6708
Epoch 4/5 — Train Loss: 0.6424, Train Acc: 0.6973 | Val Loss: 0.6406, Val Acc: 0.7108


In [ ]:
history_progressive = {k: history_stage1[k] + history_stage2[k] for k in history_stage1}

**Comparing all 3 variants**

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_frozen['val_acc'], label='Frozen')
plt.plot(history_finetune['val_acc'], label='Full Fine-tune')
plt.plot(history_progressive['val_acc'], label='Progressive Unfreeze')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.title('Strategy Comparison — Validation Accuracy')
plt.legend()
plt.savefig('../outputs/figures/strategy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
torch.save(model_frozen.state_dict(), '../models/model_frozen.pt')
torch.save(model_finetune.state_dict(), '../models/model_finetune.pt')
torch.save(model_progressive.state_dict(), '../models/model_progressive.pt')

#### 7. Overfitting Check — Train vs. Val (Fine-tune)
Full fine-tune was the strongest performer by validation accuracy (Section 6). Before treating it as the final model, confirming train/val accuracy stay reasonably close — a widening gap would signal overfitting rather than genuine generalization.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_finetune['train_acc'], label='Train Acc', marker='o')
plt.plot(history_finetune['val_acc'], label='Val Acc', marker='x')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Fine-tune: Train vs Val Accuracy')
plt.legend()
plt.savefig('../outputs/figures/finetune_train_vs_val.png', dpi=150, bbox_inches='tight')
plt.show()